In [1]:
import numpy as np
import pandas as pd

# from sklearn.model_selection import train_test_split, cross_val_score,
from sklearn.model_selection import train_test_split, GroupKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.dummy import DummyRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import warnings
warnings.filterwarnings("ignore")


### LOADING THE DATA

In [3]:
data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_df.csv")

In [4]:
data.head()

,subject,test_time,subject.1,test_time.1,motor_updrs,jitter,shimmer,hnr,dfa,rpde,...,log_ppe,log_hnr,dysphonia_index,age_jitter_interaction,age_shimmer_interaction,delta_lag_1,time_diff,delta_trend_3,baseline_updrs,delta_motor_updrs
0,1,12.666,1,12.666,28.447,0.00300,0.02024,27.183,0.56477,0.43493,...,0.102647,3.338719,-0.421653,0.21600,1.45728,0.000,7.0229,0.124000,28.199,0.248
1,1,19.681,1,19.681,28.695,0.00481,0.01675,23.047,0.54405,0.46222,...,0.190736,3.180010,0.561632,0.34632,1.20600,0.248,7.0150,0.248000,28.199,0.496
2,1,25.647,1,25.647,28.905,0.00528,0.02309,24.445,0.57794,0.48730,...,0.287259,3.236519,1.084348,0.38016,1.66248,0.496,5.9660,0.483333,28.199,0.706
3,1,33.642,1,33.642,29.187,0.00335,0.01703,26.126,0.56122,0.47188,...,0.176982,3.300493,-0.045860,0.24120,1.22616,0.706,7.9950,0.730000,28.199,0.988
4,1,40.652,1,40.652,29.435,0.00353,0.02227,22.946,0.57243,0.53949,...,0.178146,3.175801,0.436311,0.25416,1.60344,0.988,7.0100,0.976667,28.199,1.236


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5833 entries, 0 to 5832
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   subject                  5833 non-null   int64  
 1   test_time                5833 non-null   float64
 2   subject.1                5833 non-null   int64  
 3   test_time.1              5833 non-null   float64
 4   motor_updrs              5833 non-null   float64
 5   jitter                   5833 non-null   float64
 6   shimmer                  5833 non-null   float64
 7   hnr                      5833 non-null   float64
 8   dfa                      5833 non-null   float64
 9   rpde                     5833 non-null   float64
 10  ppe                      5833 non-null   float64
 11  age                      5833 non-null   int64  
 12  sex                      5833 non-null   int64  
 13  voice_instability_mean   5833 non-null   float64
 14  voice_instability_std   

In [6]:
data["updrs_lag1"] = data.groupby("subject")["motor_updrs"].shift(1)


#### Create a new lagged feature by grouping the patients(subjects) and accessing the "motor_updrs" column within each group.
#### Shift the values down by 1 within each group

In [8]:
data = data.dropna(subset=["updrs_lag1"]).reset_index(drop=True)

#### Drop null values associated with the lagged feature created

In [10]:
data["updrs_lag1"].isna().sum()

0

In [11]:
target = "delta_motor_updrs"

features = [
    "baseline_updrs",
    "jitter",
    "shimmer",
    "hnr",
    "rpde",
    "ppe",
    "sex",
    "updrs_lag1"
]


#### Create the target anf features to be used

In [13]:
X = data[features]
y = data[target]

In [14]:
groups = data["subject"] 


#### Create a group using the subjects for the GroupKFold

In [16]:
unique_subjects = data["subject"].unique()

train_subjects, test_subjects = train_test_split(
    unique_subjects, test_size=0.2, random_state=42
)

train_mask = data["subject"].isin(train_subjects)
test_mask = data["subject"].isin(test_subjects)

X_train = X[train_mask]
X_test  = X[test_mask]

y_train = y[train_mask]
y_test  = y[test_mask]

# Get patient IDs for each split (useful for grouping/cross-validation)
g_train = data.loc[train_mask, "subject"]
g_test  = data.loc[test_mask, "subject"]


#### Do the train test split of the data. The split is done subject-wise and not the normal row-wise splitting

In [18]:
models = {
    "Dummy_Mean": DummyRegressor(strategy="mean"),
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
}


#### Train the models that are suitable to work with the data

In [20]:
cv = GroupKFold(n_splits=5)


#### Performing the cross-validation group K fold and also creating the metrics to measure the models

In [22]:
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}


In [23]:
results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        groups=g_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
    "model": name,
    "MAE": -scores["test_mae"].mean(),
    "RMSE": -scores["test_rmse"].mean(),
    "R2": scores["test_r2"].mean()
})




In [24]:
print(scores.keys())


dict_keys(['fit_time', 'score_time', 'test_mae', 'test_rmse', 'test_r2'])


In [25]:
baseline_results = pd.DataFrame(results).sort_values("RMSE")
baseline_results

,model,MAE,RMSE,R2
2,Ridge,0.624868,1.280251,0.889467
1,Linear,0.628967,1.281243,0.889270
3,RandomForest,1.633390,2.466822,0.545237
0,Dummy_Mean,2.995771,3.944956,-0.020998


In [26]:
best_model_name = baseline_results.iloc[0]["model"]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

Ridge()

In [27]:
y_pred = best_model.predict(X_test)

In [28]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [29]:


print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:",rmse)
print("Test R2:", r2_score(y_test, y_pred))


Test MAE: 0.6434604128813897
Test RMSE: 1.4947037648242563
Test R2: 0.8708674260326509


In [30]:
print("Train variance:", np.var(y_train))
print("Test variance:", np.var(y_test))

Train variance: 16.009302438369613
Test variance: 17.30112918793596


In [31]:
print(y_test.min(), y_test.max())
print(y_pred.min(), y_pred.max())

-13.117999999999997 12.9259
-12.38919949991588 12.553318097468994


In [32]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

y_dummy = dummy.predict(X_test)

print("Dummy R2:", r2_score(y_test, y_dummy))
print("Model R2:", r2_score(y_test, y_pred))

Dummy R2: -0.08113601994707031
Model R2: 0.8708674260326509


In [33]:
np.std(y_pred), np.std(y_test)


(3.9462140844950295, 4.159462608070418)

#### This confrims that the predictions are not trivial

In [35]:
direction_acc = np.mean(
    np.sign(y_test) == np.sign(y_pred)
)
direction_acc


0.8200493015612161

#### Checking direction accuracy

In [37]:
df_eval = data.loc[X_test.index, ["subject"]].copy()
df_eval["y_true"] = y_test.values
df_eval["y_pred"] = y_pred

df_eval.groupby("subject").apply(
    lambda x: mean_absolute_error(x["y_true"], x["y_pred"])
).describe()


count    9.000000
mean     0.662415
std      0.487084
min      0.224764
25%      0.247538
50%      0.420866
75%      1.113442
max      1.442421
dtype: float64

In [38]:
coef_df = pd.Series(
    best_model.coef_,
    index=X_train.columns
).sort_values()
coef_df


baseline_updrs   -0.951449
rpde             -0.540888
ppe              -0.398169
jitter           -0.268701
hnr              -0.002420
sex               0.044611
updrs_lag1        0.950043
shimmer           1.236461
dtype: float64

In [39]:
baseline_results.to_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/baseline_model_results.csv", index=False)

In [40]:
# data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_df.csv")
# data.head()
# target = "delta_motor_updrs"

# features = [
#     "jitter",
#     "shimmer",
#     "hnr",
#     "rpde",
#     "ppe",
#     "sex"
# ]



# groups = data["test_time"] 
# X = data[features]
# y = data[target]

# X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
#     X, y, groups,
#     test_size=0.2, random_state=42,
#     shuffle=False
# )

# models = {
#     "Dummy_Mean": DummyRegressor(strategy="mean"),
#     "Linear": LinearRegression(),
#     "Ridge": Ridge(alpha=1.0),
#     "RandomForest": RandomForestRegressor(
#         n_estimators=200,
#         max_depth=10,
#         random_state=42,
#         n_jobs=-1
#     )
# }


# cv = GroupKFold(n_splits=5)

# scoring = {
#     "mae": "neg_mean_absolute_error",
#     "rmse": "neg_root_mean_squared_error",
#     "r2": "r2"
# }


# scoring = {
#     "mae": "neg_mean_absolute_error",
#     "rmse": "neg_root_mean_squared_error",
#     "r2": "r2"
# }

# results = []

# for name, model in models.items():
#     scores = cross_validate(
#         model,
#         X_train,
#         y_train,
#         groups=g_train,
#         cv=cv,
#         scoring=scoring,
#         n_jobs=-1
#     )

#     results.append({
#     "model": name,
#     "MAE": -scores["test_mae"].mean(),
#     "RMSE": -scores["test_rmse"].mean(),
#     "R2": scores["test_r2"].mean()
# })


# baseline_results = pd.DataFrame(results).sort_values("RMSE")
# baseline_results

# best_model_name = baseline_results.iloc[0]["model"]
# best_model = models[best_model_name]

# best_model.fit(X_train, y_train)

# y_pred = best_model.predict(X_test)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))


# print("Test MAE:", mean_absolute_error(y_test, y_pred))
# print("Test RMSE:",rmse)
# print("Test R2:", r2_score(y_test, y_pred))


In [41]:
# data.columns

In [42]:
# target = "motor_updrs"


In [43]:
# X = data[features]
# y = data[target]

In [44]:
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
# models = {
#     "Linear Regression": LinearRegression(),
#     "Ridge": Ridge(alpha=0.1),
#     "Lasso": Lasso(alpha=0.1),
#     "Elastic Net": ElasticNet(alpha=0.1, l1_ratio=0.5),
#     "SVR": SVR(kernel='rbf', C=10, epsilon=0.1),
#     "Random Forest Regressor": RandomForestRegressor(n_estimators=300, max_depth=12, 
#                                 min_samples_leaf=5, random_state=42, n_jobs=-1),
#     "Gradient Boost Regressor": GradientBoostingRegressor(
#         n_estimators=300,
#         learning_rate=0.05,
#         max_depth=3,
#         random_state=42
#     )


# }

### CREATING THE TRAINING AND EVALUATION LOOP

In [47]:
# results = []

# for name, model in models.items():
#     pipe = Pipeline([
#         ("Scaler", StandardScaler()),
#         ("model", model)
#     ])
#     pipe.fit(X_train, y_train)
#     preds = pipe.predict(X_val)

#     results.append({
#         "model": name,
#         "MAE": mean_absolute_error(y_val, preds),
#         "RMSE": mean_squared_error(y_val, preds) **0.5,
#         "R2 Score": r2_score(y_val, preds)
#     })
# results_data = pd.DataFrame(results).sort_values("RMSE")
# results_data